## SVM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, recall_score, make_scorer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.metrics import precision_recall_curve

from pipeline_and_scorer import get_shared_preprocessing_pipeline, cost_function, custom_scorer

X_train = pd.read_csv('../data/split/X_train.csv')
y_train = pd.read_csv('../data/split/y_train.csv').squeeze()
X_test = pd.read_csv('../data/split/X_test.csv')
y_test = pd.read_csv('../data/split/y_test.csv').squeeze()

print(f"Rozmiar: {X_train.shape}")
print(f"Rozkład klas:\n{y_train.value_counts(normalize=True)}")

Rozmiar: (72668, 4)
Rozkład klas:
hazardous
0    0.902681
1    0.097319
Name: proportion, dtype: float64


In [14]:
preprocessing = get_shared_preprocessing_pipeline()

baseline_pipe = ImbPipeline(preprocessing.steps + [
    ('model', SVC(class_weight='balanced', random_state=42))
])

baseline_pipe.fit(X_train, y_train)
y_pred_base = baseline_pipe.predict(X_train)

print("Baseline SVC")
print(classification_report(y_train, y_pred_base))

Baseline SVC
              precision    recall  f1-score   support

           0       1.00      0.74      0.85     65596
           1       0.29      1.00      0.45      7072

    accuracy                           0.76     72668
   macro avg       0.64      0.87      0.65     72668
weighted avg       0.93      0.76      0.81     72668



Strojenie paramtrów

In [15]:
param_grid = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['linear', 'rbf'],
    'model__gamma': ['scale', 'auto'],
    'model__class_weight': ['balanced', {0: 1, 1: 10}, {0: 1, 1: 100}] 
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=baseline_pipe,
    param_grid=param_grid,
    scoring=custom_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Najlepsze parametry: {grid_search.best_params_}")
y_pred_grid = grid_search.predict(X_train)
print(classification_report(y_train, y_pred_grid))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Najlepsze parametry: {'model__C': 0.1, 'model__class_weight': {0: 1, 1: 100}, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
              precision    recall  f1-score   support

           0       1.00      0.69      0.82     65596
           1       0.26      1.00      0.41      7072

    accuracy                           0.72     72668
   macro avg       0.63      0.84      0.61     72668
weighted avg       0.93      0.72      0.78     72668



In [27]:
best_model =  grid_search.best_estimator_

y_scores = best_model.decision_function(X_train)

precisions, recalls, thresholds = precision_recall_curve(y_train, y_scores)

safe_indices = np.where(recalls >= 0.9999)[0] 
best_idx = safe_indices[-1] if len(safe_indices) > 0 else 0
safe_threshold = thresholds[best_idx]

print(f"Nowy threshold: {safe_threshold:.4f}")
y_pred_safe = (y_scores >= safe_threshold).astype(int)
print(classification_report(y_train, y_pred_safe))

Nowy threshold: 0.4433
              precision    recall  f1-score   support

           0       1.00      0.72      0.83     65596
           1       0.28      1.00      0.43      7072

    accuracy                           0.74     72668
   macro avg       0.64      0.86      0.63     72668
weighted avg       0.93      0.74      0.80     72668



In [24]:
final_model_base = grid_search.best_estimator_
safe_threshold_base = safe_threshold

y_scores_test_base = final_model_base.decision_function(X_test)
y_pred_test_safe_base = (y_scores_test_base >= safe_threshold_base).astype(int)

print("Ewaluacja modelu bez SMOTE:")
print(classification_report(y_test, y_pred_test_safe_base))

Ewaluacja modelu bez SMOTE:
              precision    recall  f1-score   support

           0       1.00      0.72      0.84     16400
           1       0.28      1.00      0.43      1768

    accuracy                           0.75     18168
   macro avg       0.64      0.86      0.63     18168
weighted avg       0.93      0.75      0.80     18168



In [17]:
smote_pipe = ImbPipeline(preprocessing.steps + [
    ('smote', SMOTE(random_state=42)),
    ('model', SVC(random_state=42))
])

param_grid_smote = {
    'model__C': [1, 10],
    'model__kernel': ['rbf'],
    'smote__k_neighbors': [3, 5]
}

grid_smote = GridSearchCV(smote_pipe, param_grid_smote, scoring=custom_scorer, cv=cv, n_jobs=-1)
grid_smote.fit(X_train, y_train)

print(f"Najlepsze parametry SMOTE: {grid_smote.best_params_}")
y_pred_smote = grid_smote.predict(X_train)
print(classification_report(y_train, y_pred_smote))

Najlepsze parametry SMOTE: {'model__C': 1, 'model__kernel': 'rbf', 'smote__k_neighbors': 3}
              precision    recall  f1-score   support

           0       1.00      0.74      0.85     65596
           1       0.29      1.00      0.45      7072

    accuracy                           0.76     72668
   macro avg       0.64      0.87      0.65     72668
weighted avg       0.93      0.76      0.81     72668



In [28]:
best_model = grid_smote.best_estimator_ 
y_scores = best_model.decision_function(X_train)
precisions, recalls, thresholds = precision_recall_curve(y_train, y_scores)

safe_indices = np.where(recalls >= 0.9999)[0] 
best_idx = safe_indices[-1] if len(safe_indices) > 0 else 0
safe_threshold = thresholds[best_idx]

print(f"Nowy threshold: {safe_threshold:.4f}")

y_pred_safe = (y_scores >= safe_threshold).astype(int)
print(classification_report(y_train, y_pred_safe))

Nowy threshold: -0.7182
              precision    recall  f1-score   support

           0       1.00      0.70      0.82     65596
           1       0.26      1.00      0.42      7072

    accuracy                           0.73     72668
   macro avg       0.63      0.85      0.62     72668
weighted avg       0.93      0.73      0.78     72668



In [26]:
final_model = grid_smote.best_estimator_ 

safe_threshold = thresholds[best_idx] 

y_scores_test = final_model.decision_function(X_test)
y_pred_test_safe = (y_scores_test >= safe_threshold).astype(int)


print("Ewaluacja modelu SMOTE:")
print(classification_report(y_test, y_pred_test_safe))

Ewaluacja modelu SMOTE:
              precision    recall  f1-score   support

           0       1.00      0.70      0.82     16400
           1       0.27      1.00      0.42      1768

    accuracy                           0.73     18168
   macro avg       0.63      0.85      0.62     18168
weighted avg       0.93      0.73      0.78     18168



## Wnioski
Wszystkie modele poprawnie stanowią ochronę zachowując 0 fn. Najleszą opcją okazał się bazowy  SVM trenowany na zbalansowanych klasami, uzyskując recall 72% klasy 0 dla zbioru testowego. Hipreparametry które koazały się najnardziej skkuteczne przy tej metodzie to:
- C: 0.1 
- model__class_weight': {0: 1, 1: 100} 
- gamma: auto 
- kernel: rbf
- threshold: 0.4433
